# 构建三个不同风格空间的高 PMI 值关键词

## 加载风格空间（训练集）

In [ ]:
from pathlib import Path
from typing import List
import json

MUICE_PATH = Path("../Dataset/Muice/train.jsonl")
HARUHI_PATH = Path(r"..\Dataset\Haruhi\Haruhi_clean.jsonl")
PSYDC_PATH1 = Path(r"..\Dataset\PsyDTCorpus\PsyDTCorpus_train_mulit_turn_packing.json")
PSYDC_PATH2 = Path(r"..\Dataset\PsyDTCorpus\PsyDTCorpus_test_single_turn_split.json")

# Check if Dataset exist
assert MUICE_PATH.is_file(), "请确保 Muice Dataset 的路径正确！"
assert HARUHI_PATH.is_file(), "请确保 Haruhi Dataset 的路径正确！"
assert PSYDC_PATH1.is_file(), "请确保 PsyDTCorpus Dataset 的路径正确！"

# Load Muice Dataset
dataset_lines = MUICE_PATH.read_text(encoding="utf-8").splitlines()

muice_responses: List[str] = []

for line in dataset_lines:
    item = json.loads(line)
    muice_responses.append(item["Response"])

# 取 80% 作为训练集
muice_responses = muice_responses[:int(len(muice_responses)*0.8)]

# Load Haruhi Dataset
dataset_lines = HARUHI_PATH.read_text(encoding="utf-8").splitlines()

haruhi_responses: List[str] = []

for line in dataset_lines:
    item = json.loads(line)

    # 只提取单一角色的训练集
    if item["agent_role"] == "神里绫华":
        haruhi_responses.append(item["agent_response"])

# 取 70 %作为训练集
haruhi_responses = haruhi_responses[:int(len(haruhi_responses)*0.7)]


# Load PsyDTCorpus Dataset
psydc_part1 = json.loads(PSYDC_PATH1.read_text(encoding="utf-8"))
psydc_part2 = json.loads(PSYDC_PATH2.read_text(encoding="utf-8"))
psydc: list[dict] = psydc_part1 + psydc_part2
psydc_responses: List[str] = []

for item in psydc:
    messages: list[dict[str, str]] = item["messages"]
    for index in range(2, len(messages), 2):
        message = messages[index]
        assert message["role"] == "assistant", message
        psydc_responses.append(message["content"])

# 取 90 %作为训练集
psydc_responses = psydc_responses[:int(len(psydc_responses)*0.7)]

# 输出所有训练集的长度

print(f"Muice Responses: {len(muice_responses)}")
print(f"Haruhi Responses: {len(haruhi_responses)}")
print(f"PsyDTCorpus Responses: {len(psydc_responses)}")

Muice Responses: 2721
Haruhi Responses: 991
PsyDTCorpus Responses: 89652


## 调用 HanLP 进行分词

In [ ]:
from hanlp_restful import HanLPClient
from time import sleep
import re

HanLP = HanLPClient('https://www.hanlp.com/api', auth="ODc1OEBiYnMuaGFubHAuY29tOlN5WVFNMFJrNnlqcm4yaGo=", language='zh')

def limit_sentense_length(text: str, max_length: int = 150) -> list[str]:
    if len(text) < 150:
        return [text]
    
    # print("Warning: 句子超出了长度上限: ", text)

    # 按照标点符号进行分割
    sentences = re.split(r'([。！？；，,.!?;])', text)
    chunks:list[str] = []
    temp = ''
    for i in range(0, len(sentences), 2):
        chunk = sentences[i] + (sentences[i+1] if i+1 < len(sentences) else '')
        if len(temp) + len(chunk) < max_length:
            temp += chunk
        else:
            chunks.append(temp)
            temp = chunk
    if temp:
        chunks.append(temp)
    return chunks

def split_texts(texts: list[str]) -> list[str]:
    """将文本列表中的每个文本分割成更小的句子"""
    all_sentences = []
    for text in texts:
        sentences = limit_sentense_length(text)
        all_sentences.extend(sentences)
    return all_sentences

def tokenize_safe(texts: list[str], max_batch_num: int = 250, max_chars_per_batch: int = 15000, interval: int = 35):
    """对文本进行分词，同时限制每一批总字符数"""
    all_tokens = []
    current_batch = []
    current_length = 0
    batch_id = 1

    for text in texts:
        text_len = len(text)
        # 如果加上这个句子会超出限制，则先处理已有批次
        if current_length + text_len > max_chars_per_batch or len(current_batch) + 1 > max_batch_num:
            print(f"Processing batch {batch_id} (Total chars: {current_length})...")
            tokens = HanLP.tokenize(current_batch)
            all_tokens.extend(tokens)
            sleep(interval)
            batch_id += 1
            # 重置 batch
            current_batch = [text]
            current_length = text_len
        else:
            current_batch.append(text)
            current_length += text_len

    # 最后一批也别忘记
    if current_batch:
        print(f"Processing batch {batch_id} (Total chars: {current_length})...")
        tokens = HanLP.tokenize(current_batch)
        all_tokens.extend(tokens)

    return all_tokens

muice_responses_split = split_texts(muice_responses)
haruhi_responses_split = split_texts(haruhi_responses)
psydc_responses_split = split_texts(psydc_responses)

muice_tokens = tokenize_safe(muice_responses_split, 240, interval=20)
haruhi_tokens = tokenize_safe(haruhi_responses_split, 200, interval=20)
psydc_tokens = tokenize_safe(psydc_responses_split, 200, interval=20)

Processing batch 1 (Total chars: 10009)...
Processing batch 2 (Total chars: 14951)...
Processing batch 3 (Total chars: 14981)...
Processing batch 4 (Total chars: 14967)...
Processing batch 5 (Total chars: 14893)...
Processing batch 6 (Total chars: 10435)...
Processing batch 1 (Total chars: 11259)...
Processing batch 2 (Total chars: 11822)...
Processing batch 3 (Total chars: 10710)...
Processing batch 4 (Total chars: 12365)...
Processing batch 5 (Total chars: 12210)...
Processing batch 6 (Total chars: 11664)...
Processing batch 7 (Total chars: 12171)...
Processing batch 8 (Total chars: 11598)...
Processing batch 9 (Total chars: 11546)...
Processing batch 10 (Total chars: 11472)...
Processing batch 11 (Total chars: 11889)...
Processing batch 12 (Total chars: 12252)...
Processing batch 13 (Total chars: 12717)...
Processing batch 14 (Total chars: 11952)...
Processing batch 15 (Total chars: 12603)...
Processing batch 16 (Total chars: 11940)...
Processing batch 17 (Total chars: 11203)...
Pro

In [ ]:
# 扁平化处理并存档
from itertools import chain
import pickle

# print(muice_tokens[:50])

muice_tokens: list[str] = list(chain.from_iterable(muice_tokens))
haruhi_tokens: list[str] = list(chain.from_iterable(haruhi_tokens))
psydc_tokens: list[str] = list(chain.from_iterable(psydc_tokens))

# print(f"Muice Tokens: {len(muice_tokens)}")
print(f"Haruhi Tokens: {len(haruhi_tokens)}")
print(f"PsyDTCorpus Tokens: {len(psydc_tokens)}")

with open("muice_tokens.pkl", "wb") as f:
    pickle.dump(muice_tokens, f)
with open("haruhi_tokens.pkl", "wb") as f:
    pickle.dump(haruhi_tokens, f)
with open("psydc_tokens.pkl", "wb") as f:
    pickle.dump(psydc_tokens, f)


Haruhi Tokens: 51134
PsyDTCorpus Tokens: 3417372


## downsampling 处理

由于风格语料库之间的差异比约为 1:1:64 ，若直接应用于 PMI 计算会导致严重的误差。

因此我们将使用一种名为下采样(downsampling)的方法以优化差异比。

在以下的代码块中，我们将使用下采样，将差异比缩减为 1:1:4

In [33]:
# Load tokens from pkl(如果之前已经运行过了就可以注释此段)

# import pickle
# muice_tokens = pickle.load(open("muice_tokens.pkl", "rb"))
# haruhi_tokens = pickle.load(open("haruhi_tokens.pkl", "rb"))
# psydc_tokens = pickle.load(open("psydc_tokens.pkl", "rb"))

print("Before downsampling:")
print(f"Muice Tokens: {len(muice_tokens)}")
print(f"Haruhi Tokens: {len(haruhi_tokens)}")
print(f"PsyDTCorpus Tokens: {len(psydc_tokens)}")

import random

# Downsample tokens to a target size

def downsample(tokens: list[str], target_size: int) -> list[str]:
    """对 tokens 进行下采样，直到达到目标大小"""
    if len(tokens) <= target_size:
        return tokens
    return random.sample(tokens, target_size)

muice_tokens_downsampled = muice_tokens
haruhi_tokens_downsampled = haruhi_tokens
psydc_tokens_downsampled = downsample(psydc_tokens, len(haruhi_tokens) * 4)
global_tokens_downsampled = muice_tokens_downsampled + haruhi_tokens_downsampled + psydc_tokens_downsampled

print("-"* 20)

print("After downsampling:")
print(f"Muice Tokens: {len(muice_tokens_downsampled)}")
print(f"Haruhi Tokens: {len(haruhi_tokens_downsampled)}")
print(f"PsyDTCorpus Tokens: {len(psydc_tokens_downsampled)}")
print(f"Global Tokens: {len(global_tokens_downsampled)}")


Before downsampling:
Muice Tokens: 53829
Haruhi Tokens: 51134
PsyDTCorpus Tokens: 3417372
--------------------
After downsampling:
Muice Tokens: 53829
Haruhi Tokens: 51134
PsyDTCorpus Tokens: 204536
Global Tokens: 309499


## 计算不同风格的 PMI 风格词汇表

为了忽略特定主题的词语和较少出现的词，当这个词的 $p(w) > 10%$ 或 $p(w|t) < 0.5%$ 时我们将其忽略不计。我们对三个不同的风格领域中的词元进行了 PMI 计算并挑选出具有代表性的风格词元。


In [ ]:
import math
from collections import Counter
import unicodedata

def pmi(w:str, style_counter: Counter, global_style_counter: Counter) -> float:
    """
    计算词 w 在风格 t 下的 PMI 值
    """
    pw_t = style_counter[w] / sum(style_counter.values()) if style_counter[w] > 0 else 1e-9
    pw = global_style_counter[w] / sum(global_style_counter.values()) if global_style_counter else 1e-9
    return math.log(pw_t / pw, 2) if pw > 0 else float('-inf')  # 避免除以零的情况

EXTRA_PUNCTUATION = set("⋯…～·—“”‘’<>")

def is_punctuation(word: str) -> bool:
    return all(
        unicodedata.category(char).startswith("P") or char in EXTRA_PUNCTUATION
        for char in word
    )

def filter_pmi(pmi_dict: dict[str, float], min_pmi: float = 0.3, max_pmi: float = 10.0) -> dict[str, float]:
    """
    筛选 PMI 词汇表，保留符合条件的词
    """
    filtered = {}

    for word, pmi_value in pmi_dict.items():
        # 按出现频率筛选
        if not (min_pmi <= pmi_value <= max_pmi):
            continue
        # 忽略单个字符的词
        # elif len(word) < 2:
        #     continue
        # 去除标点符号
        elif is_punctuation(word):
            continue

        filtered[word] = pmi_value

    return filtered

muice_collecter = Counter(muice_tokens_downsampled)
haruhi_collecter = Counter(haruhi_tokens_downsampled)
psydc_collecter = Counter(psydc_tokens_downsampled)
global_collecter = Counter(global_tokens_downsampled)

# 计算每个风格的 PMI 词汇表
muice_pmi = {word: pmi(word, muice_collecter, global_collecter) for word in muice_collecter}
haruhi_pmi = {word: pmi(word, haruhi_collecter, global_collecter) for word in haruhi_collecter}
psydc_pmi = {word: pmi(word, psydc_collecter, global_collecter) for word in psydc_collecter}

# 筛选词汇表
muice_pmi_filtered = filter_pmi(muice_pmi)
haruhi_pmi_filtered = filter_pmi(haruhi_pmi)
psydc_pmi_filtered = filter_pmi(psydc_pmi)

print(f"Muice PMI Vocabulary Set Size: {len(set(muice_pmi_filtered))}")
print(f"Haruhi PMI Vocabulary Set Size: {len(set(haruhi_pmi_filtered))}")
print(f"PsyDTCorpus PMI Vocabulary Set Size: {len(set(psydc_pmi_filtered))}")

# 排序词汇表
muice_pmi_filtered = dict(sorted(muice_pmi_filtered.items(), key=lambda item: item[1], reverse=True))
haruhi_pmi_filtered = dict(sorted(haruhi_pmi_filtered.items(), key=lambda item: item[1], reverse=True))
psydc_pmi_filtered = dict(sorted(psydc_pmi_filtered.items(), key=lambda item: item[1], reverse=True))

print("Muice PMI Vocabulary Sample:")
print(list(muice_pmi_filtered.keys())[:25])
print("Haruhi PMI Vocabulary Sample:")
print(list(haruhi_pmi_filtered.keys())[:25])
print("PsyDTCorpus PMI Vocabulary Sample:")
print(list(psydc_pmi_filtered.keys())[:25])

# 保存词汇表

# with open("muice_pmi_filtered.json", "w", encoding="utf-8") as f:
#     json.dump(muice_pmi_filtered, f, ensure_ascii=False, indent=4)
# with open("haruhi_pmi_filtered.json", "w", encoding="utf-8") as f:
#     json.dump(haruhi_pmi_filtered, f, ensure_ascii=False, indent=4)
# with open("psydc_pmi_filtered.json", "w", encoding="utf-8") as f:
#     json.dump(psydc_pmi_filtered, f, ensure_ascii=False, indent=4)

Muice PMI Vocabulary Set Size: 4726
Haruhi PMI Vocabulary Set Size: 2670
PsyDTCorpus PMI Vocabulary Set Size: 2445
Muice PMI Vocabulary Sample:
['奇奇怪怪', '天真', '钛合金', '沐沐', '傻', '女孩子', '兴', '就说', '沐雪', '雪雪', '⭐', '开发', '16', '岁', '身高', '喵', '天大', '人家', '摸鱼', '讨厌', 'USERNAME', '死', '库水', '裸露', '浅粉色']
Haruhi PMI Vocabulary Sample:
['神里家', '传奇', '坚守', '稻妻国', '将军', '传', '锻', '术', '星象', '矿', '质', '炉火', '即是', '雷电', '锻造', '名刀', '刀工', '算作', '本领', '归属', '统筹', '祭祀', '奉行', '一派', '督办']
PsyDTCorpus PMI Vocabulary Sample:
['例如', '心理学', '孤立', '那时', '人之常情', '短期', '有帮助', '投入', '男朋友', '平衡点', '赞扬', '咨询师', '很多时候', '挫败感', '亲密', '设立', '理清', '念头', '小组', '会议', '转化', '亲近', '反而', '预期', '安抚']


: 

In [5]:
import json

# Read results
with open("muice_pmi_filtered.json", "r", encoding="utf-8") as f:
    muice_pmi_filtered = json.load(f)
with open("haruhi_pmi_filtered.json", "r", encoding="utf-8") as f:
    haruhi_pmi_filtered = json.load(f)
with open("psydc_pmi_filtered.json", "r", encoding="utf-8") as f:
    psydc_pmi_filtered = json.load(f)

print(f"Muice PMI Vocabulary Set Size: {len(set(muice_pmi_filtered))}")
print(f"Haruhi PMI Vocabulary Set Size: {len(set(haruhi_pmi_filtered))}")
print(f"PsyDTCorpus PMI Vocabulary Set Size: {len(set(psydc_pmi_filtered))}")

# print(f"Top-300 Muice PMI Vocabulary: {list(muice_pmi_filtered.keys())[:300]}")
print(f"Top-300 Haruhi PMI Vocabulary: {list(haruhi_pmi_filtered.keys())[:300]}")
print(f"Top-300 PsyDTCorpus PMI Vocabulary: {' '.join(list(psydc_pmi_filtered.keys())[:300])}")

Muice PMI Vocabulary Set Size: 4726
Haruhi PMI Vocabulary Set Size: 2670
PsyDTCorpus PMI Vocabulary Set Size: 2445
Top-300 Haruhi PMI Vocabulary: ['神里家', '传奇', '坚守', '稻妻国', '将军', '传', '锻', '术', '星象', '矿', '质', '炉火', '即是', '雷电', '锻造', '名刀', '刀工', '算作', '本领', '归属', '统筹', '祭祀', '奉行', '一派', '督办', '不力', '仿佛', '片刻', '亲缘', '割舍', '祭典', '实', '相瞒', '久利须', '收下', '意', '稍事', '花园', '边上', '景点', '一事', '助', '我个人', '神里', '委托', '神社', '石碑', '赐予', '名叫', '眼', '情报', '家主', '兄长', '平日', '出面', '在身', '唉', '当是', '劝言', '看完', '记载', '神使', '椿', '派蒙', '坂', '制造', '狩', '呵呵，', '过奖', '浪人', '匆忙', '来不及', '丢失', '货物', '敬重', '民众', '谋', '是为', '稻妻', '大御所', '剑', '修行', '邪', '静心', '斩断', '纷争', '沉淀', '二者', '迥然不同', '白鹭', '公主', '神里绫华', '风和日丽', '哪边', '周遭', '神道', '幕后', '势力', '华胥引', '诗句', '幻境', '陶醉', '残酷', '阶层', '民生', '遭到', '闲话', '裟罗', '眼狩令', '执行者', '听从', '正胜', '彼此彼此', '办到', '救出', '礼节', '体恤', '良善', '可靠', '稍等', '付钱', '咳', '定做', '准确性', '木南', '这家', '餐厅', '菜品', '特地', '前来', '呼啊', '风儿', '清爽', '微风', '拂', '洗净', '尘埃', '霁', '银', '妆', '素', '桔', '映', 